In [1]:

from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from dotenv import load_dotenv
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()
model=HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V3",
    task="text-generation"
)

llm=ChatHuggingFace(llm=model)

/Users/devgupta/Desktop/LangGraph/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str


In [6]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}


In [7]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}


In [8]:

graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)


In [9]:

config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)


{'topic': 'pizza',
 'joke': "Sure! Here's a cheesy pizza joke for you:  \n\n**Why did the pizza maker go to therapy?**  \nBecause he couldn't stop *unraveling* his deep-dish issues! 🍕😆  \n\nLet me know if you want another slice of humor!",
 'explanation': 'Here\'s a breakdown of why this pizza joke is funny:\n\n1. **Wordplay on "unraveling"**: The humor hinges on the double meaning of "unraveling."  \n   - Literal meaning: Pizza makers often *stretch/unravel* dough when preparing it.  \n   - Figurative meaning: In therapy, people *unravel* (explore/process) their emotional issues.  \n\n2. **"Deep-dish issues" pun**:  \n   - A *deep-dish pizza* is a thick, Chicago-style pizza.  \n   - "Deep issues" refers to serious personal problems—here, it’s mashed up with pizza terminology for comedic effect.  \n\n3. **Unexpected twist**: The setup suggests a serious scenario (therapy), but the punchline absurdly ties it to pizza-making, subverting expectations.  \n\n4. **Visual humor**: The emojis 

In [10]:
workflow.get_state(config1)


StateSnapshot(values={'topic': 'pizza', 'joke': "Sure! Here's a cheesy pizza joke for you:  \n\n**Why did the pizza maker go to therapy?**  \nBecause he couldn't stop *unraveling* his deep-dish issues! 🍕😆  \n\nLet me know if you want another slice of humor!", 'explanation': 'Here\'s a breakdown of why this pizza joke is funny:\n\n1. **Wordplay on "unraveling"**: The humor hinges on the double meaning of "unraveling."  \n   - Literal meaning: Pizza makers often *stretch/unravel* dough when preparing it.  \n   - Figurative meaning: In therapy, people *unravel* (explore/process) their emotional issues.  \n\n2. **"Deep-dish issues" pun**:  \n   - A *deep-dish pizza* is a thick, Chicago-style pizza.  \n   - "Deep issues" refers to serious personal problems—here, it’s mashed up with pizza terminology for comedic effect.  \n\n3. **Unexpected twist**: The setup suggests a serious scenario (therapy), but the punchline absurdly ties it to pizza-making, subverting expectations.  \n\n4. **Visual h

In [11]:
list(workflow.get_state_history(config1))


[StateSnapshot(values={'topic': 'pizza', 'joke': "Sure! Here's a cheesy pizza joke for you:  \n\n**Why did the pizza maker go to therapy?**  \nBecause he couldn't stop *unraveling* his deep-dish issues! 🍕😆  \n\nLet me know if you want another slice of humor!", 'explanation': 'Here\'s a breakdown of why this pizza joke is funny:\n\n1. **Wordplay on "unraveling"**: The humor hinges on the double meaning of "unraveling."  \n   - Literal meaning: Pizza makers often *stretch/unravel* dough when preparing it.  \n   - Figurative meaning: In therapy, people *unravel* (explore/process) their emotional issues.  \n\n2. **"Deep-dish issues" pun**:  \n   - A *deep-dish pizza* is a thick, Chicago-style pizza.  \n   - "Deep issues" refers to serious personal problems—here, it’s mashed up with pizza terminology for comedic effect.  \n\n3. **Unexpected twist**: The setup suggests a serious scenario (therapy), but the punchline absurdly ties it to pizza-making, subverting expectations.  \n\n4. **Visual 

In [12]:

config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)


{'topic': 'pasta',
 'joke': "Sure! Here's a pasta joke for you:  \n\n**Why did the spaghetti break up with the fettuccine?**  \nBecause it found its partner *too al dente*!  \n\nHope that gives you a noodle chuckle! 🍝😄 Let me know if you'd like another one!",
 'explanation': 'Here\'s a breakdown of why this pasta joke is funny:\n\n1. **Wordplay on "al dente"**: The term *al dente* (Italian for "to the tooth") describes pasta cooked to be firm when bitten. But in the joke, it’s humorously reinterpreted as a personality trait—suggesting the fettuccine is *too rigid* or *inflexible* in the relationship, just like undercooked pasta.  \n\n2. **Anthropomorphism**: The joke gives human traits to noodles (spaghetti and fettuccine "breaking up"), turning a mundane cooking term into a playful relationship drama.  \n\n3. **Unexpected twist**: The punchline subverts expectations—you might anticipate a food-related answer, but the joke cleverly ties it to romantic incompatibility.  \n\n4. **Pasta p

In [14]:

workflow.get_state(config2)


StateSnapshot(values={'topic': 'pasta', 'joke': "Sure! Here's a pasta joke for you:  \n\n**Why did the spaghetti break up with the fettuccine?**  \nBecause it found its partner *too al dente*!  \n\nHope that gives you a noodle chuckle! 🍝😄 Let me know if you'd like another one!", 'explanation': 'Here\'s a breakdown of why this pasta joke is funny:\n\n1. **Wordplay on "al dente"**: The term *al dente* (Italian for "to the tooth") describes pasta cooked to be firm when bitten. But in the joke, it’s humorously reinterpreted as a personality trait—suggesting the fettuccine is *too rigid* or *inflexible* in the relationship, just like undercooked pasta.  \n\n2. **Anthropomorphism**: The joke gives human traits to noodles (spaghetti and fettuccine "breaking up"), turning a mundane cooking term into a playful relationship drama.  \n\n3. **Unexpected twist**: The punchline subverts expectations—you might anticipate a food-related answer, but the joke cleverly ties it to romantic incompatibility

In [15]:
list(workflow.get_state_history(config2))


[StateSnapshot(values={'topic': 'pasta', 'joke': "Sure! Here's a pasta joke for you:  \n\n**Why did the spaghetti break up with the fettuccine?**  \nBecause it found its partner *too al dente*!  \n\nHope that gives you a noodle chuckle! 🍝😄 Let me know if you'd like another one!", 'explanation': 'Here\'s a breakdown of why this pasta joke is funny:\n\n1. **Wordplay on "al dente"**: The term *al dente* (Italian for "to the tooth") describes pasta cooked to be firm when bitten. But in the joke, it’s humorously reinterpreted as a personality trait—suggesting the fettuccine is *too rigid* or *inflexible* in the relationship, just like undercooked pasta.  \n\n2. **Anthropomorphism**: The joke gives human traits to noodles (spaghetti and fettuccine "breaking up"), turning a mundane cooking term into a playful relationship drama.  \n\n3. **Unexpected twist**: The punchline subverts expectations—you might anticipate a food-related answer, but the joke cleverly ties it to romantic incompatibilit

Time Travel

In [29]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f173b12-b91f-66a2-8000-b8d7b520505f"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f173b12-b91f-66a2-8000-b8d7b520505f'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-06-29T11:53:40.258768+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f173b12-b91d-6bf4-bfff-a4ef1fb0c43e'}}, tasks=(PregelTask(id='8aed6c8c-5a4e-ce6d-5f70-858890a4138b', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': "Sure! Here's a cheesy pizza joke for you:  \n\n**Why did the pizza maker go to therapy?**  \nBecause he couldn't stop *unraveling* his deep-dish issues! 🍕😆  \n\nLet me know if you want another slice of humor!"}),), interrupts=())

In [30]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f173b12-b91f-66a2-8000-b8d7b520505f"}})

{'topic': 'pizza',
 'joke': "Sure! Here's a cheesy pizza joke for you:  \n\n🍕 **Why did the pizza maker go to therapy?**  \nBecause he couldn’t stop *dough*-pressing his feelings!  \n\nHope that gave you a *slice* of laughter! 😄🍕",
 'explanation': 'Here’s a breakdown of why this pizza joke works so well:\n\n1. **Pun on "Dough" and "Depressing"**:  \n   The humor hinges on the wordplay between *dough* (the base ingredient of pizza) and *depressing* (feeling sad or overwhelmed). The pizza maker is literally pressing dough (kneading it to make pizza), but figuratively, he’s "dough-pressing" (depressing) his emotions—hence needing therapy.\n\n2. **Occupation-Themed Twist**:  \n   Jokes about professions (like bakers, doctors, etc.) are fun because they tie the punchline to the person’s daily work. Here, the pizza maker’s job (handling dough) becomes a metaphor for emotional struggles.\n\n3. **Cheesy Double Meaning**:  \n   The joke leans into the idea of "cheesy" humor (pun intended), wher

In [31]:

list(workflow.get_state_history(config1))


[StateSnapshot(values={'topic': 'pizza', 'joke': "Sure! Here's a cheesy pizza joke for you:  \n\n🍕 **Why did the pizza maker go to therapy?**  \nBecause he couldn’t stop *dough*-pressing his feelings!  \n\nHope that gave you a *slice* of laughter! 😄🍕", 'explanation': 'Here’s a breakdown of why this pizza joke works so well:\n\n1. **Pun on "Dough" and "Depressing"**:  \n   The humor hinges on the wordplay between *dough* (the base ingredient of pizza) and *depressing* (feeling sad or overwhelmed). The pizza maker is literally pressing dough (kneading it to make pizza), but figuratively, he’s "dough-pressing" (depressing) his emotions—hence needing therapy.\n\n2. **Occupation-Themed Twist**:  \n   Jokes about professions (like bakers, doctors, etc.) are fun because they tie the punchline to the person’s daily work. Here, the pizza maker’s job (handling dough) becomes a metaphor for emotional struggles.\n\n3. **Cheesy Double Meaning**:  \n   The joke leans into the idea of "cheesy" humor 

Updating State

In [37]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f173b12-b91f-66a2-8000-b8d7b520505f", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f173b35-67fc-6cc0-8001-ac869f6a85d7'}}

In [38]:

list(workflow.get_state_history(config1))


[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f173b35-67fc-6cc0-8001-ac869f6a85d7'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-06-29T12:09:11.275226+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f173b12-b91f-66a2-8000-b8d7b520505f'}}, tasks=(PregelTask(id='552121a3-f65b-9cc0-ebc2-181efd7f50f7', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': "Sure! Here's a cheesy pizza joke for you:  \n\n**Why did the pizza maker go to art school?**  \nBecause he wanted to *master the crust*! 🍕🎨  \n\nHope that gave you a *slice* of laughter! 😄", 'explanation': 'Here\'s a breakdown of why this pizza joke works:\n\n1. **Wordplay on "Master the Crust":**  \n   - *Master the crust* soun

In [39]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f173b35-67fc-6cc0-8001-ac869f6a85d7"}})

{'topic': 'samosa',
 'joke': 'Sure! Here\'s a samosa joke for you:  \n\n**Why did the samosa go to the gym?**  \nBecause it wanted to lose its *extra filling*! 😆  \n\n(Get it? Like "extra filling" as in the stuffing inside, but also like trying to lose weight? 😉)  \n\nHope that brings a smile! Want another one?',
 'explanation': 'Here’s a breakdown of why this samosa joke works:  \n\n1. **Wordplay on "filling"**:  \n   - A samosa is a fried pastry filled with spiced potatoes, peas, or meat. The term *"extra filling"* literally refers to the delicious stuffing inside it.  \n   - But in fitness/health contexts, *"losing extra filling"* sounds like losing excess weight (like shedding fat or calories).  \n\n2. **Unexpected Gym Motivation**:  \n   - Samosas are indulgent, calorie-rich snacks—so the idea of one "working out" is absurd and funny. It personifies the samosa as if it’s health-conscious.  \n\n3. **Cultural Relatability**:  \n   - Many people joke about balancing their love for sa

In [40]:
list(workflow.get_state_history(config1))


[StateSnapshot(values={'topic': 'samosa', 'joke': 'Sure! Here\'s a samosa joke for you:  \n\n**Why did the samosa go to the gym?**  \nBecause it wanted to lose its *extra filling*! 😆  \n\n(Get it? Like "extra filling" as in the stuffing inside, but also like trying to lose weight? 😉)  \n\nHope that brings a smile! Want another one?', 'explanation': 'Here’s a breakdown of why this samosa joke works:  \n\n1. **Wordplay on "filling"**:  \n   - A samosa is a fried pastry filled with spiced potatoes, peas, or meat. The term *"extra filling"* literally refers to the delicious stuffing inside it.  \n   - But in fitness/health contexts, *"losing extra filling"* sounds like losing excess weight (like shedding fat or calories).  \n\n2. **Unexpected Gym Motivation**:  \n   - Samosas are indulgent, calorie-rich snacks—so the idea of one "working out" is absurd and funny. It personifies the samosa as if it’s health-conscious.  \n\n3. **Cultural Relatability**:  \n   - Many people joke about balanci

Fault Tolerance

In [16]:

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [22]:

# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str
    step3: str

In [23]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}


In [19]:

# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)


In [21]:

try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")


▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
❌ Kernel manually interrupted (crash simulated).


In [ ]:

# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state) 



🔁 Re-running the graph to demonstrate fault tolerance...
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


KeyboardInterrupt: 

In [24]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f173b20-7322-6da8-8004-54c268a78ae8'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-06-29T11:59:48.729788+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f173b20-7321-6606-8003-36a34bc557a0'}}, tasks=(PregelTask(id='d0ea3b04-a20a-9be4-6303-441dbb4ae3d9', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_1',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f173b20-7321-6606-8003-36a34bc557a0'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-06-29T11:59:48.729186+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkp